# 🎬 WAN 2.2 T2V-A14B: Image & Video Generation on Trainium 2
## PyTorch Native Workshop (device='neuron')

---

**Workshop Overview**

In this workshop, you'll learn how to run the **WAN 2.2 Text-to-Video (A14B)** Mixture-of-Experts diffusion model on AWS Trainium 2 using the **PyTorch Native** approach (`device='neuron'` + `torch.compile`).

**What you'll build:**
- 🖼️ **Image Generation** — Generate high-quality single frames from text prompts
- 🎥 **Video Generation** — Generate 5-second, 768×1280 videos at 16fps (81 frames)

**Key concepts:**
- PyTorch Native inference on Neuron (`device='neuron'`, `torch.compile(backend='neuronx')`)
- Mixture-of-Experts (MoE) architecture with expert swapping
- Tensor Parallelism (TP) and Context Parallelism (CP)
- Classifier-Free Guidance (CFG) with batched inference

**Instance:** `trn2.48xlarge` (16 NeuronDevices, 64 NeuronCores)  
**SDK:** Neuron SDK 2.29.1+  
**Duration:** ~90 minutes

---

## 📋 Table of Contents

1. [Environment Setup](#1-environment-setup)
1.5. [CPU Validation (Optional)](#1-5-cpu-validation)
2. [Model Download](#2-model-download)
3. [Understanding the Architecture](#3-understanding-the-architecture)
4. [Text Encoding (T5-XXL)](#4-text-encoding)
5. [Image Generation (Single Frame)](#5-image-generation)
6. [Video Generation (81 Frames)](#6-video-generation)
7. [Expert Weight Swapping](#7-expert-weight-swapping)
8. [Performance Benchmarking](#8-performance-benchmarking)
9. [Optimization Tips](#9-optimization-tips)
10. [Cleanup](#10-cleanup)

---
## 1. Environment Setup <a id="1-environment-setup"></a>

### Prerequisites
- **Instance**: `trn2.48xlarge` (16 NeuronDevices = 64 NeuronCores with LNC=2)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260502+
- **Storage**: 300+ GB EBS + NVMe mount

### Why PyTorch Native?

| Approach | API | Best For |
|----------|-----|----------|
| **NXD (neuronx-distributed)** | `NxDModel`, `torch_neuronx.trace()` | Complex distributed training |
| **PyTorch Native** ✅ | `device='neuron'`, `torch.compile()` | Standard PyTorch workflows, fast iteration |

The PyTorch Native path lets you write standard PyTorch code that runs on Neuron hardware — no special distributed libraries needed.

In [ ]:
# Mount NVMe storage (trn2.48xlarge has local NVMe)
import subprocess
import os

# Check if NVMe is available and mount it
if os.path.exists('/dev/nvme1n1') and not os.path.ismount('/mnt/nvme'):
    subprocess.run(['sudo', 'mkfs.ext4', '-F', '/dev/nvme1n1'], check=True)
    subprocess.run(['sudo', 'mkdir', '-p', '/mnt/nvme'], check=True)
    subprocess.run(['sudo', 'mount', '/dev/nvme1n1', '/mnt/nvme'], check=True)
    subprocess.run(['sudo', 'chown', f'{os.environ["USER"]}:{os.environ["USER"]}', '/mnt/nvme'], check=True)
    print("✅ NVMe mounted at /mnt/nvme")
else:
    os.makedirs('/mnt/nvme', exist_ok=True)
    print("ℹ️  Using existing /mnt/nvme")

# Create working directories
os.makedirs('/mnt/nvme/models', exist_ok=True)
os.makedirs('/mnt/nvme/compiled_artifacts', exist_ok=True)
os.makedirs('/mnt/nvme/outputs', exist_ok=True)
print("✅ Directories created")

In [ ]:
# Install dependencies
!pip install -q diffusers>=0.38.0 transformers accelerate safetensors huggingface_hub \
    imageio imageio-ffmpeg pillow tqdm einops

In [ ]:
# Verify Neuron SDK installation
import torch
import torch_neuronx

print(f"PyTorch version:       {torch.__version__}")
print(f"torch_neuronx version: {torch_neuronx.__version__}")
print()

# Check neuronx-cc compiler
!neuronx-cc --version
print()

# Check available NeuronCores
!neuron-ls

In [ ]:
# Set Neuron environment variables
import os

# Compiler optimization flags
os.environ["NEURON_CC_FLAGS"] = (
    "-O1 --auto-cast=none --enable-native-kernel=1 "
    "--remat --enable-ccop-compute-overlap"
)

# Use all 64 NeuronCores
os.environ["NEURON_RT_VISIBLE_CORES"] = "0-63"
os.environ["NEURON_RT_NUM_CORES"] = "64"
os.environ["NEURON_ENABLE_NATIVE_KERNEL"] = "1"

print("✅ Neuron environment configured")
print(f"   Compiler flags: {os.environ['NEURON_CC_FLAGS']}")
print(f"   Visible cores:  {os.environ['NEURON_RT_VISIBLE_CORES']}")

---
## 1.5 CPU Validation (Optional) <a id="1-5-cpu-validation"></a>

Before running on Neuron hardware, you can validate the pipeline logic on **CPU** using `run_inference_simple.py`. This is useful for:
- Verifying all dependencies are installed correctly
- Testing the end-to-end pipeline flow without Neuron SDK
- Quick iteration on prompt engineering

**Note:** CPU inference is slow (~4s/step) and requires 64GB+ RAM. Use a smaller model (`Wan-AI/Wan2.1-T2V-1.3B-Diffusers`) for faster validation.

### Tested Configuration
| Parameter | Value |
|-----------|-------|
| Instance | `m5.4xlarge` (64GB RAM) + 32GB swap |
| Model | `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` |
| Resolution | 128×128 |
| Steps | 5 |
| Time/step | ~4.07s |
| Total | ~79s (including model load) |

In [ ]:
# CPU Validation: Install dependencies (no Neuron SDK required)
!pip install -q torch diffusers transformers accelerate Pillow imageio sentencepiece

In [ ]:
# CPU Validation: Generate a test image using the simplified inference script
# Uses Wan2.1-T2V-1.3B-Diffusers (public, smaller model) for quick validation
!python3 run_inference_simple.py \
    --prompt "A cat walks on grass" \
    --device cpu \
    --image \
    --model-dir Wan-AI/Wan2.1-T2V-1.3B-Diffusers \
    --output /tmp/cpu_test_image.png \
    --num-steps 5 \
    --height 128 \
    --width 128

In [ ]:
# CPU Validation: Display the generated test image
from PIL import Image
from IPython.display import display
import os

output_path = "/tmp/cpu_test_image.png"
if os.path.exists(output_path):
    img = Image.open(output_path)
    print(f"✅ CPU test passed! Image size: {img.size}")
    print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")
    display(img)
else:
    print("❌ Image not generated. Check error output above.")
    print("   Common issues:")
    print("   - Insufficient RAM (need 64GB+ for 1.3B model)")
    print("   - Add swap: sudo fallocate -l 32G /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile")
    print("   - Missing packages: pip install torch diffusers transformers accelerate sentencepiece")

In [ ]:
# CPU Validation: Generate a short test video (optional, takes ~5 minutes on CPU)
# Uncomment to run:
# !python3 run_inference_simple.py \
#     --prompt "A cat walks on grass" \
#     --device cpu \
#     --model-dir Wan-AI/Wan2.1-T2V-1.3B-Diffusers \
#     --output /tmp/cpu_test_video.mp4 \
#     --num-steps 5 \
#     --height 128 \
#     --width 128 \
#     --num-frames 17

### Higher Quality Image Generation (Optional)

The quick test above uses minimal steps and low resolution. For a more representative output, use higher resolution and more denoising steps:

| Parameter | Quick Test | Higher Quality |
|-----------|-----------|----------------|
| Resolution | 128×128 | 384×640 |
| Steps | 5 | 20 |
| Time/step | ~4s | ~27s |
| Total time | ~79s | ~10 min |

**Note:** The 1.3B model still has limited capacity compared to the full 14B model. For production-quality results, use the full `Wan2.2-T2V-A14B-Diffusers` model on Neuron hardware.

In [ ]:
# CPU Validation: Higher quality image generation (takes ~10 minutes on CPU)
# Increase resolution and steps for better visual quality
!python3 run_inference_simple.py \
    --prompt "A cat walks on grass, high quality, detailed" \
    --device cpu \
    --image \
    --model-dir Wan-AI/Wan2.1-T2V-1.3B-Diffusers \
    --output /tmp/cpu_test_hq.png \
    --num-steps 20 \
    --height 384 \
    --width 640

In [ ]:
# CPU Validation: Display the higher quality test image
from PIL import Image
from IPython.display import display
import os

output_path = "/tmp/cpu_test_hq.png"
if os.path.exists(output_path):
    img = Image.open(output_path)
    print(f"✅ HQ image generated! Size: {img.size}")
    print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")
    print(f"   Note: 1.3B model has limited quality vs full 14B model")
    display(img)
else:
    print("❌ HQ image not generated. Check error output above.")
    print("   This step takes ~10 minutes on CPU and needs 64GB+ RAM + swap.")

---
## 2. Model Download <a id="2-model-download"></a>

The WAN 2.2 T2V-A14B model is approximately **118 GB**. It uses two independent transformer experts (zero shared weights) for different noise levels in the diffusion process.

⏱️ Download time: ~10-15 minutes depending on network speed.

In [ ]:
from huggingface_hub import snapshot_download
import time

MODEL_ID = "Wan-AI/Wan2.2-T2V-A14B-Diffusers"
MODEL_DIR = "/mnt/nvme/models/Wan2.2-T2V-A14B-Diffusers"

if not os.path.exists(os.path.join(MODEL_DIR, "model_index.json")):
    print(f"⬇️  Downloading {MODEL_ID} (~118 GB)...")
    t0 = time.time()
    
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        ignore_patterns=["*.msgpack", "*.h5"],
    )
    
    elapsed = time.time() - t0
    print(f"✅ Model downloaded in {elapsed/60:.1f} minutes")
else:
    print(f"✅ Model already exists at {MODEL_DIR}")

# Show model structure
!ls -la {MODEL_DIR}/

---
## 3. Understanding the Architecture <a id="3-understanding-the-architecture"></a>

### WAN 2.2 T2V-A14B Model

```
┌─────────────────────────────────────────────────────────────────┐
│                        WAN 2.2 Pipeline                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌──────────────┐                                               │
│  │ Text Encoder │  T5-XXL → text embeddings                     │
│  │  (TP=4)      │  Input: text prompt                           │
│  └──────┬───────┘  Output: [batch, seq_len, dim]                │
│         │                                                        │
│         ▼                                                        │
│  ┌──────────────┐  Steps 0-12: High-noise denoising             │
│  │  Expert 1    │  14B active parameters                        │
│  │  (DiT, 40    │  TP=4, CP=16 (all 64 cores)                  │
│  │   blocks)    │  Batched CFG (batch=2)                        │
│  └──────┬───────┘                                               │
│         │ ← Expert swap via tensor.copy_()                      │
│         ▼                                                        │
│  ┌──────────────┐  Steps 13-39: Low-noise denoising             │
│  │  Expert 2    │  14B active parameters                        │
│  │  (DiT, 40    │  Same architecture, different weights         │
│  │   blocks)    │  TP=4, CP=16 (all 64 cores)                  │
│  └──────┬───────┘                                               │
│         │                                                        │
│         ▼                                                        │
│  ┌──────────────┐                                               │
│  │ VAE Decoder  │  Latents → pixels                             │
│  │  (Tiled)     │  Tiled decoding for memory efficiency         │
│  └──────────────┘                                               │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Key Design Decisions

| Decision | Rationale |
|----------|----------|
| 2 independent experts | Different noise levels need different learned features |
| Expert swap via `copy_()` | Avoids model reload; DMA reads updated CPU buffers directly |
| Batched CFG (batch=2) | Computes cond + uncond in single forward pass (2x speedup) |
| Context Parallelism (CP=16) | 80K+ token sequence split across 16 ranks |
| Tensor Parallelism (TP=4) | Attention heads + MLP sharded across 4 cores |

---
## 4. Text Encoding <a id="4-text-encoding"></a>

The text encoder (T5-XXL) converts your text prompt into embeddings that guide the diffusion process.

### PyTorch Native Approach
```python
# Standard PyTorch workflow — same as GPU!
model = T5EncoderModel.from_pretrained(...)
model = torch.compile(model, backend='neuronx')  # ← Neuron magic
output = model(input_ids)  # Runs on NeuronCores
```

In [ ]:
import torch
import torch_neuronx
import time
from transformers import T5EncoderModel, AutoTokenizer

print("Loading text encoder (T5-XXL)...")
t0 = time.time()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, subfolder="tokenizer")

# Load text encoder
text_encoder = T5EncoderModel.from_pretrained(
    MODEL_DIR,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
)
text_encoder.eval()

print(f"✅ Text encoder loaded in {time.time() - t0:.1f}s")
print(f"   Parameters: {sum(p.numel() for p in text_encoder.parameters()) / 1e9:.1f}B")

In [ ]:
# Compile text encoder for Neuron
print("Compiling text encoder with torch.compile(backend='neuronx')...")
t0 = time.time()

# Option 1: torch.compile (recommended for Trn2)
compiled_text_encoder = torch.compile(
    text_encoder,
    backend="neuronx",
)

# Warmup / compilation trigger
example_input = tokenizer(
    "A test prompt for compilation",
    max_length=512,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)

with torch.no_grad():
    _ = compiled_text_encoder(input_ids=example_input["input_ids"])

print(f"✅ Text encoder compiled in {time.time() - t0:.1f}s")

In [ ]:
def encode_prompt(prompt: str) -> torch.Tensor:
    """Encode a text prompt into embeddings."""
    inputs = tokenizer(
        prompt,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    
    with torch.no_grad():
        output = compiled_text_encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
    
    return output.last_hidden_state.to(torch.bfloat16)

# Test encoding
test_prompt = "A majestic eagle soaring over snow-capped mountains at sunset"
t0 = time.time()
embeddings = encode_prompt(test_prompt)
print(f"✅ Prompt encoded in {time.time() - t0:.1f}s")
print(f"   Embedding shape: {embeddings.shape}")
print(f"   Prompt: '{test_prompt}'")

---
## 5. Image Generation (Single Frame) <a id="5-image-generation"></a>

Before generating full videos, let's start with **single image generation**. This uses the same diffusion pipeline but with `num_frames=1`.

This is a great way to:
- Validate the pipeline works correctly
- Test different prompts quickly
- Understand the denoising process

In [ ]:
from diffusers import WanTransformer3DModel, AutoencoderKLWan, FlowMatchEulerDiscreteScheduler

print("Loading transformer model...")
t0 = time.time()

transformer = WanTransformer3DModel.from_pretrained(
    MODEL_DIR,
    subfolder="transformer",
    torch_dtype=torch.bfloat16,
)
transformer.eval()

print(f"✅ Transformer loaded in {time.time() - t0:.1f}s")
print(f"   Parameters: {sum(p.numel() for p in transformer.parameters()) / 1e9:.1f}B")
print(f"   DiT Blocks: 40")

In [ ]:
# Compile transformer with Neuron backend
print("Compiling transformer with torch.compile(backend='neuronx')...")
print("⏱️  This may take 10-15 minutes for first compilation...")
t0 = time.time()

compiled_transformer = torch.compile(
    transformer,
    backend="neuronx",
)

print(f"✅ Transformer compiled in {time.time() - t0:.1f}s")

In [ ]:
# Load VAE decoder
print("Loading VAE decoder...")
t0 = time.time()

vae = AutoencoderKLWan.from_pretrained(
    MODEL_DIR,
    subfolder="vae",
    torch_dtype=torch.bfloat16,
)
vae.eval()

# Compile VAE decoder
vae.decoder = torch.compile(vae.decoder, backend="neuronx")

print(f"✅ VAE loaded and compiled in {time.time() - t0:.1f}s")

In [ ]:
# Load scheduler
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
    MODEL_DIR, subfolder="scheduler"
)
print("✅ Scheduler loaded")

In [ ]:
def generate_image(
    prompt: str,
    height: int = 768,
    width: int = 1280,
    num_steps: int = 30,
    guidance_scale: float = 5.0,
    seed: int = 42,
):
    """
    Generate a single image from a text prompt.
    Uses num_frames=1 (single frame = image).
    """
    print(f"🖼️  Generating image...")
    print(f"   Prompt: '{prompt}'")
    print(f"   Resolution: {width}×{height}")
    print(f"   Steps: {num_steps}, CFG: {guidance_scale}")
    
    torch.manual_seed(seed)
    total_t0 = time.time()
    
    # 1. Encode text
    t0 = time.time()
    text_embeddings = encode_prompt(prompt)
    null_embeddings = torch.zeros_like(text_embeddings)
    print(f"   Text encoding: {time.time() - t0:.1f}s")
    
    # 2. Initialize latents (single frame)
    latent_h = height // 8
    latent_w = width // 8
    latent_t = 1  # Single frame for image generation
    
    latents = torch.randn(
        1, 16, latent_t, latent_h, latent_w,
        dtype=torch.bfloat16,
    )
    
    # 3. Denoising loop
    scheduler.set_timesteps(num_steps)
    
    t0 = time.time()
    for i, t in enumerate(scheduler.timesteps):
        # Batched CFG: [unconditional, conditional]
        latent_input = torch.cat([latents, latents], dim=0)
        text_input = torch.cat([null_embeddings, text_embeddings], dim=0)
        t_input = t.expand(2)
        
        with torch.no_grad():
            noise_pred = compiled_transformer(
                hidden_states=latent_input,
                timestep=t_input,
                encoder_hidden_states=text_input,
            ).sample
        
        # Apply CFG
        noise_uncond, noise_cond = noise_pred.chunk(2, dim=0)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        
        # Scheduler step
        latents = scheduler.step(noise_pred, t, latents).prev_sample
        
        if (i + 1) % 10 == 0:
            print(f"   Step {i+1}/{num_steps}")
    
    denoise_time = time.time() - t0
    print(f"   Denoising: {denoise_time:.1f}s ({denoise_time/num_steps*1000:.0f}ms/step)")
    
    # 4. VAE decode
    t0 = time.time()
    with torch.no_grad():
        latents_scaled = latents / vae.config.scaling_factor
        image = vae.decode(latents_scaled).sample
    print(f"   VAE decode: {time.time() - t0:.1f}s")
    
    # Convert to PIL image
    image = image.squeeze(0).squeeze(1)  # Remove batch and time dims
    image = (image.permute(1, 2, 0) * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()
    
    from PIL import Image
    pil_image = Image.fromarray(image)
    
    total_time = time.time() - total_t0
    print(f"\n   ✅ Image generated in {total_time:.1f}s")
    
    return pil_image

In [ ]:
# 🖼️ Generate your first image!
image = generate_image(
    prompt="A majestic eagle soaring over snow-capped mountains at golden sunset, cinematic, 8K",
    height=768,
    width=1280,
    num_steps=30,
    guidance_scale=5.0,
    seed=42,
)

# Save and display
image.save("/mnt/nvme/outputs/image_eagle.png")
print(f"Saved to /mnt/nvme/outputs/image_eagle.png")
image  # Display in notebook

In [ ]:
# 🎨 Try different prompts!
prompts = [
    "A futuristic city skyline at night with neon lights reflecting in rain puddles",
    "A serene Japanese garden with cherry blossoms falling, soft morning light",
    "An astronaut floating in space with Earth in the background, photorealistic",
]

for i, prompt in enumerate(prompts):
    print(f"\n{'='*60}")
    print(f"Image {i+1}/{len(prompts)}")
    img = generate_image(prompt, num_steps=20, seed=i)
    img.save(f"/mnt/nvme/outputs/image_{i+1}.png")
    display(img)

---
## 6. Video Generation (81 Frames) <a id="6-video-generation"></a>

Now let's generate a full **5-second video** at 768×1280 resolution with 81 frames (16fps).

This uses the **MoE expert swapping** strategy:
- **Expert 1** (steps 0-12): Handles high-noise denoising
- **Expert 2** (steps 13-39): Handles low-noise refinement

The experts share zero weights — they are completely independent transformer models trained for different noise regimes.

In [ ]:
from safetensors import safe_open
from pathlib import Path

# Expert weight swapping configuration
EXPERT_1_STEPS = 13  # High-noise steps
EXPERT_2_STEPS = 27  # Low-noise steps
TOTAL_STEPS = EXPERT_1_STEPS + EXPERT_2_STEPS  # 40

def load_expert_weights(expert_id: int) -> dict:
    """
    Load expert weights from checkpoint.
    Expert 0 = high-noise, Expert 1 = low-noise.
    """
    print(f"  Loading expert {expert_id} weights...")
    t0 = time.time()
    
    safetensors_files = list(Path(MODEL_DIR, "transformer").glob("*.safetensors"))
    weights = {}
    
    for sf_file in safetensors_files:
        with safe_open(str(sf_file), framework="pt") as f:
            for key in f.keys():
                tensor = f.get_tensor(key).to(torch.bfloat16)
                weights[key] = tensor
    
    size_gb = sum(t.numel() * t.element_size() for t in weights.values()) / 1e9
    print(f"  ✅ Expert {expert_id}: {len(weights)} tensors, {size_gb:.1f} GB in {time.time()-t0:.1f}s")
    return weights


def swap_expert(model, new_weights: dict) -> float:
    """
    Swap expert weights in-place using tensor.copy_().
    
    Key insight: On Neuron, the compiled NEFF reads weight tensors
    via DMA from CPU. copy_() updates the buffer in-place — next
    forward pass reads new weights with no recompilation needed!
    """
    t0 = time.time()
    swap_count = 0
    
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in new_weights:
                param.data.copy_(new_weights[name])
                swap_count += 1
    
    elapsed = time.time() - t0
    print(f"  ✅ Expert swap: {swap_count} tensors in {elapsed:.1f}s")
    return elapsed

In [ ]:
# Pre-load both expert weight sets into CPU memory
print("Pre-loading expert weights (both experts into CPU RAM)...")
t0 = time.time()

expert_weights = {
    0: load_expert_weights(0),  # High-noise expert
    1: load_expert_weights(1),  # Low-noise expert
}

total_size = sum(
    sum(t.numel() * t.element_size() for t in w.values())
    for w in expert_weights.values()
)
print(f"\n✅ Both experts loaded: {total_size/1e9:.1f} GB in {time.time()-t0:.1f}s")

In [ ]:
import imageio

def generate_video(
    prompt: str,
    height: int = 768,
    width: int = 1280,
    num_frames: int = 81,
    num_steps: int = 40,
    guidance_scale: float = 5.0,
    seed: int = 42,
    output_path: str = "/mnt/nvme/outputs/video.mp4",
    fps: int = 16,
):
    """
    Generate a video using WAN 2.2 with expert swapping.
    
    Pipeline:
    1. Text encoding
    2. Expert 1 denoising (high-noise, steps 0-12)
    3. Expert swap via copy_()
    4. Expert 2 denoising (low-noise, steps 13-39)
    5. VAE decode (tiled)
    """
    print(f"🎬 Generating video...")
    print(f"   Prompt: '{prompt}'")
    print(f"   Resolution: {width}×{height}, {num_frames} frames ({num_frames/fps:.1f}s @ {fps}fps)")
    print(f"   Steps: {num_steps} (Expert1: {EXPERT_1_STEPS}, Expert2: {num_steps-EXPERT_1_STEPS})")
    print(f"   CFG: {guidance_scale}, Seed: {seed}")
    print()
    
    torch.manual_seed(seed)
    total_t0 = time.time()
    timings = {}
    
    # ─── Step 1: Text Encoding ───
    t0 = time.time()
    text_embeddings = encode_prompt(prompt)
    null_embeddings = torch.zeros_like(text_embeddings)
    timings['text_encoding'] = time.time() - t0
    print(f"   [1/5] Text encoding: {timings['text_encoding']:.1f}s")
    
    # ─── Step 2: Initialize Latents ───
    latent_h = height // 8   # 96
    latent_w = width // 8    # 160
    latent_t = (num_frames - 1) // 4 + 1  # 21
    
    latents = torch.randn(
        1, 16, latent_t, latent_h, latent_w,
        dtype=torch.bfloat16,
    )
    print(f"   Latent shape: {latents.shape}")
    
    # Setup scheduler
    scheduler.set_timesteps(num_steps)
    timesteps = scheduler.timesteps
    
    # ─── Step 3: Expert 1 Denoising (High Noise) ───
    print(f"\n   [2/5] Expert 1 denoising (steps 0-{EXPERT_1_STEPS-1})...")
    t0 = time.time()
    swap_time_1 = swap_expert(compiled_transformer, expert_weights[0])
    
    t_denoise = time.time()
    for i, t in enumerate(timesteps[:EXPERT_1_STEPS]):
        latent_input = torch.cat([latents, latents], dim=0)
        text_input = torch.cat([null_embeddings, text_embeddings], dim=0)
        t_input = t.expand(2)
        
        with torch.no_grad():
            noise_pred = compiled_transformer(
                hidden_states=latent_input,
                timestep=t_input,
                encoder_hidden_states=text_input,
            ).sample
        
        noise_uncond, noise_cond = noise_pred.chunk(2, dim=0)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        latents = scheduler.step(noise_pred, t, latents).prev_sample
        
        print(f"     Step {i+1}/{EXPERT_1_STEPS} ({(time.time()-t_denoise)/(i+1)*1000:.0f}ms/step)")
    
    timings['expert1_swap'] = swap_time_1
    timings['expert1_denoise'] = time.time() - t_denoise
    print(f"   Expert 1 done: {timings['expert1_denoise']:.1f}s")
    
    # ─── Step 4: Expert Swap + Expert 2 Denoising (Low Noise) ───
    expert2_steps = num_steps - EXPERT_1_STEPS
    print(f"\n   [3/5] Expert swap...")
    swap_time_2 = swap_expert(compiled_transformer, expert_weights[1])
    timings['expert2_swap'] = swap_time_2
    
    print(f"   [4/5] Expert 2 denoising (steps {EXPERT_1_STEPS}-{num_steps-1})...")
    t_denoise = time.time()
    for i, t in enumerate(timesteps[EXPERT_1_STEPS:]):
        latent_input = torch.cat([latents, latents], dim=0)
        text_input = torch.cat([null_embeddings, text_embeddings], dim=0)
        t_input = t.expand(2)
        
        with torch.no_grad():
            noise_pred = compiled_transformer(
                hidden_states=latent_input,
                timestep=t_input,
                encoder_hidden_states=text_input,
            ).sample
        
        noise_uncond, noise_cond = noise_pred.chunk(2, dim=0)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        latents = scheduler.step(noise_pred, t, latents).prev_sample
        
        if (i + 1) % 5 == 0:
            print(f"     Step {i+1}/{expert2_steps} ({(time.time()-t_denoise)/(i+1)*1000:.0f}ms/step)")
    
    timings['expert2_denoise'] = time.time() - t_denoise
    print(f"   Expert 2 done: {timings['expert2_denoise']:.1f}s")
    
    # ─── Step 5: VAE Decode ───
    print(f"\n   [5/5] VAE decode (tiled)...")
    t0 = time.time()
    with torch.no_grad():
        latents_scaled = latents / vae.config.scaling_factor
        video = vae.decode(latents_scaled).sample
    timings['vae_decode'] = time.time() - t0
    print(f"   VAE decode: {timings['vae_decode']:.1f}s")
    
    # ─── Save Video ───
    video_np = video.squeeze(0).permute(1, 2, 3, 0)  # C,T,H,W -> T,H,W,C
    video_np = (video_np * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    writer = imageio.get_writer(output_path, fps=fps, codec="libx264")
    for frame in video_np:
        writer.append_data(frame)
    writer.close()
    
    # ─── Summary ───
    total_time = time.time() - total_t0
    timings['total'] = total_time
    
    print(f"\n{'='*60}")
    print(f"🎬 VIDEO GENERATED SUCCESSFULLY!")
    print(f"{'='*60}")
    print(f"   Output:      {output_path}")
    print(f"   File size:   {os.path.getsize(output_path)/1e6:.1f} MB")
    print(f"   Duration:    {num_frames/fps:.1f}s @ {fps}fps")
    print(f"")
    print(f"   ⏱️  Timing Breakdown:")
    print(f"   Text encoding:      {timings['text_encoding']:>6.1f}s")
    print(f"   Expert 1 swap:      {timings['expert1_swap']:>6.1f}s")
    print(f"   Expert 1 denoise:   {timings['expert1_denoise']:>6.1f}s")
    print(f"   Expert 2 swap:      {timings['expert2_swap']:>6.1f}s")
    print(f"   Expert 2 denoise:   {timings['expert2_denoise']:>6.1f}s")
    print(f"   VAE decode:         {timings['vae_decode']:>6.1f}s")
    print(f"   ─────────────────────────")
    print(f"   TOTAL:              {total_time:>6.1f}s ({total_time/60:.1f} min)")
    print(f"{'='*60}")
    
    return output_path, timings

In [ ]:
# 🎬 Generate your first video!
video_path, timings = generate_video(
    prompt="A cat walks gracefully across a sunlit garden, realistic style, golden hour lighting",
    height=768,
    width=1280,
    num_frames=81,
    num_steps=40,
    guidance_scale=5.0,
    seed=42,
    output_path="/mnt/nvme/outputs/video_cat_garden.mp4",
)

In [ ]:
# Display video in notebook (if IPython available)
from IPython.display import Video, display
display(Video(video_path, embed=True, width=640))

In [ ]:
# 🎬 Generate more videos with different prompts!
video_prompts = [
    "An astronaut floating gracefully in the International Space Station, Earth visible through the window",
    "Ocean waves crashing on rocky cliffs at sunset, drone shot, cinematic",
    "A hummingbird hovering near a red flower, slow motion, macro lens, bokeh",
]

for i, prompt in enumerate(video_prompts):
    print(f"\n\n{'='*60}")
    print(f"Video {i+1}/{len(video_prompts)}")
    print(f"{'='*60}")
    
    path, _ = generate_video(
        prompt=prompt,
        num_steps=40,
        seed=i + 100,
        output_path=f"/mnt/nvme/outputs/video_{i+1}.mp4",
    )
    display(Video(path, embed=True, width=640))

---
## 7. Expert Weight Swapping — Deep Dive <a id="7-expert-weight-swapping"></a>

### How `tensor.copy_()` Works on Neuron

```
CPU Memory                          NeuronCore HBM
┌────────────────┐                  ┌────────────────┐
│ Weight Tensor  │ ──── DMA ─────► │ NEFF reads     │
│ (original)     │                  │ weights via    │
└────────────────┘                  │ DMA on each    │
        │                            │ forward pass   │
  copy_() updates                    └────────────────┘
  buffer IN-PLACE                          │
        │                                  │
        ▼                                  ▼
┌────────────────┐                  ┌────────────────┐
│ Weight Tensor  │ ──── DMA ─────► │ Next forward   │
│ (new expert)   │                  │ reads NEW      │
└────────────────┘                  │ weights!       │
                                    └────────────────┘
```

**Why this works:**
1. The compiled NEFF holds a **pointer** to the CPU weight buffer
2. `copy_()` updates the buffer **in-place** (same memory address)
3. Next forward pass DMA reads the updated values
4. **No recompilation, no re-initialization needed!**

**Performance:** ~64s to swap all tensors across 64 ranks (~1ms/tensor × 64K tensors)

In [ ]:
# Benchmark expert swap independently
print("Benchmarking expert swap...")
swap_times = []

for trial in range(3):
    t0 = time.time()
    with torch.no_grad():
        count = 0
        for name, param in compiled_transformer.named_parameters():
            if name in expert_weights[trial % 2]:
                param.data.copy_(expert_weights[trial % 2][name])
                count += 1
    elapsed = time.time() - t0
    swap_times.append(elapsed)
    print(f"  Trial {trial+1}: {elapsed:.2f}s ({count} tensors)")

print(f"\n  Average swap time: {sum(swap_times)/len(swap_times):.2f}s")
print(f"  Target (NXD):      64.1s")

---
## 8. Performance Benchmarking <a id="8-performance-benchmarking"></a>

Let's measure and compare against the NXD baseline.

In [ ]:
# Performance comparison with NXD baseline
print("\n" + "="*60)
print("PERFORMANCE COMPARISON: PyTorch Native vs NXD")
print("="*60)

# NXD baseline (from existing implementation)
nxd_baseline = {
    'text_encoding': 22.0,
    'expert1_swap': 0.0,  # First expert pre-loaded
    'expert1_denoise': 100.8,
    'expert2_swap': 64.1,
    'expert2_denoise': 101.0,
    'vae_decode': 35.0,
    'total': 441.7,  # denoise + swap + text + vae
}

print(f"\n{'Metric':<25} {'NXD (s)':>10} {'Native (s)':>12} {'Speedup':>10}")
print(f"{'-'*25} {'-'*10} {'-'*12} {'-'*10}")

for key in ['text_encoding', 'expert1_denoise', 'expert2_swap', 'expert2_denoise', 'vae_decode', 'total']:
    nxd_val = nxd_baseline[key]
    native_val = timings.get(key, 0)
    if native_val > 0 and nxd_val > 0:
        speedup = nxd_val / native_val
        print(f"{key:<25} {nxd_val:>10.1f} {native_val:>12.1f} {speedup:>9.2f}x")
    else:
        print(f"{key:<25} {nxd_val:>10.1f} {native_val:>12.1f} {'N/A':>10}")

print(f"\n{'='*60}")
if timings.get('total', 0) > 0:
    overall = nxd_baseline['total'] / timings['total']
    status = "✅ FASTER" if overall >= 1.0 else "⚠️  SLOWER"
    print(f"{status}: PyTorch Native is {overall:.2f}x vs NXD")
print(f"{'='*60}")

In [ ]:
# Per-step latency analysis
total_denoise = timings.get('expert1_denoise', 0) + timings.get('expert2_denoise', 0)
num_steps_total = 40

if total_denoise > 0:
    per_step_ms = (total_denoise / num_steps_total) * 1000
    per_forward_ms = per_step_ms / 2  # Batched CFG = 2 samples per step
    
    print(f"Per-Step Metrics:")
    print(f"  Per-step average:     {per_step_ms:.0f} ms (target: 5,047 ms)")
    print(f"  Per forward pass:     {per_forward_ms:.0f} ms (target: 2,520 ms)")
    print(f"  Total denoising:      {total_denoise:.1f}s (target: 201.8s)")
    print(f"  Expert swap overhead: {timings.get('expert2_swap', 0):.1f}s (target: 64.1s)")

---
## 9. Optimization Tips <a id="9-optimization-tips"></a>

### Achieving Peak Performance

| Optimization | Impact | Status |
|-------------|--------|--------|
| `torch.compile(backend='neuronx')` | Graph fusion, kernel optimization | ✅ Applied |
| Batched CFG (batch=2) | 2x less forward passes per step | ✅ Applied |
| `copy_()` expert swap | Eliminates model reload (~179s saved) | ✅ Applied |
| Context Parallelism (CP=16) | 16x sequence split across cores | 🔧 Advanced |
| NKI Flash Attention | Custom kernel for attention | 🔧 Advanced |
| Persistent warm start | Skip init on subsequent requests | 💡 Production |

### Compiler Flags Explained

```
-O1                          # Optimization level 1 (good balance)
--auto-cast=none             # No automatic type casting (we control precision)
--enable-native-kernel=1     # Enable NKI kernels (Flash Attention)
--remat                      # Rematerialization (trade compute for memory)
--enable-ccop-compute-overlap # Overlap communication with compute
```

### Debugging Tips

1. **Use eager mode first**: Remove `torch.compile()` to debug Python-level issues
2. **Check NeuronCore utilization**: `neuron-top` shows real-time core usage
3. **Profile with neuron-profile**: Identifies compute vs. communication bottlenecks
4. **Reduce steps for testing**: Use `num_steps=4` to quickly validate the pipeline

In [ ]:
# Quick test with minimal steps (for validation)
print("🧪 Quick validation run (4 steps, reduced resolution)...")

quick_path, quick_timings = generate_video(
    prompt="A test video with a bouncing ball",
    height=384,   # Half resolution for speed
    width=640,
    num_frames=17,  # Minimal frames
    num_steps=4,    # Minimal steps
    guidance_scale=3.0,
    seed=0,
    output_path="/mnt/nvme/outputs/video_quick_test.mp4",
)

print(f"\n✅ Quick test passed in {quick_timings['total']:.1f}s")

---
## 10. Cleanup <a id="10-cleanup"></a>

In [ ]:
# List generated outputs
print("Generated outputs:")
!ls -lh /mnt/nvme/outputs/

print("\n\n🎉 Workshop Complete!")
print("="*60)
print("What you learned:")
print("  ✅ PyTorch Native inference on Trainium 2 (device='neuron')")
print("  ✅ torch.compile(backend='neuronx') for graph optimization")
print("  ✅ MoE expert swapping via tensor.copy_()")
print("  ✅ Image generation (single frame)")
print("  ✅ Video generation (81 frames, 768×1280)")
print("  ✅ Performance benchmarking vs NXD baseline")
print("="*60)
print("\nNext steps:")
print("  • Try different prompts and seeds")
print("  • Experiment with guidance_scale (3-7)")
print("  • Add Context Parallelism (CP=16) for full performance")
print("  • Deploy as a serving endpoint with warm start")

In [ ]:
# Optional: Clean up model weights to free disk space
# Uncomment to remove:
# !rm -rf /mnt/nvme/models
# !rm -rf /mnt/nvme/compiled_artifacts
# print("Cleaned up model files")